In [6]:
# Cell 0 — Project Imports

import torch

In [7]:
# Cell 1 — Empty-Mask 상태 분류

def classify_binary_mask_pair(
    target_mask: torch.Tensor,      # [D, H, W], torch.bool
    prediction_mask: torch.Tensor,  # [D, H, W], torch.bool
) -> tuple[
    str,  # Empty-mask state
    int,  # Target foreground voxel count
    int,  # Prediction foreground voxel count
]:
    """Target·prediction binary mask의 empty 상태 분류."""

    # 비교 가능한 동일 spatial Shape인지 확인
    if target_mask.shape != prediction_mask.shape:
        raise ValueError(
            "Target과 prediction mask의 Shape가 일치해야 합니다."
        )

    # Class ID가 아닌 binary mask인지 확인
    if (
        target_mask.dtype != torch.bool
        or prediction_mask.dtype != torch.bool
    ):
        raise TypeError(
            "Target과 prediction mask는 torch.bool이어야 합니다."
        )

    # Target과 prediction의 foreground voxel 수 계산
    target_foreground_voxel_count = int(
        target_mask.sum().item()
    )

    prediction_foreground_voxel_count = int(
        prediction_mask.sum().item()
    )

    # 각 mask의 foreground 존재 여부 결정
    target_is_nonempty = (
        target_foreground_voxel_count > 0
    )

    prediction_is_nonempty = (
        prediction_foreground_voxel_count > 0
    )

    # 두 mask가 모두 비어 있는 상태 분류
    if (
        not target_is_nonempty
        and not prediction_is_nonempty
    ):
        empty_mask_state = "both_empty"

    # Target은 비었지만 prediction만 존재하는 FP-only 상태 분류
    elif (
        not target_is_nonempty
        and prediction_is_nonempty
    ):
        empty_mask_state = (
            "target_empty_prediction_nonempty"
        )

    # Target은 존재하지만 prediction이 비어 있는 FN-only 상태 분류
    elif (
        target_is_nonempty
        and not prediction_is_nonempty
    ):
        empty_mask_state = (
            "target_nonempty_prediction_empty"
        )

    # 두 mask 모두 foreground가 존재하는 일반 상태 분류
    else:
        empty_mask_state = "both_nonempty"

    return (
        empty_mask_state,
        target_foreground_voxel_count,
        prediction_foreground_voxel_count,
    )


# Empty binary mask 생성
empty_mask = torch.zeros(
    3,
    3,
    3,
    dtype=torch.bool,
)  # [D=3, H=3, W=3]


# 중앙 foreground voxel 하나를 가진 mask 생성
nonempty_mask = empty_mask.clone()
nonempty_mask[
    1,
    1,
    1,
] = True


# 네 가지 target·prediction empty 조합 구성
empty_mask_test_cases: dict[
    str,
    tuple[
        torch.Tensor,  # Target mask [D, H, W]
        torch.Tensor,  # Prediction mask [D, H, W]
    ],
] = {
    "Case A": (
        empty_mask,
        empty_mask,
    ),
    "Case B": (
        empty_mask,
        nonempty_mask,
    ),
    "Case C": (
        nonempty_mask,
        empty_mask,
    ),
    "Case D": (
        nonempty_mask,
        nonempty_mask,
    ),
}


# 각 조합의 empty-mask 상태와 foreground voxel 수 출력
for (
    case_name,
    (
        target_mask,
        prediction_mask,
    ),
) in empty_mask_test_cases.items():
    (
        empty_mask_state,
        target_voxel_count,
        prediction_voxel_count,
    ) = classify_binary_mask_pair(
        target_mask=target_mask,
        prediction_mask=prediction_mask,
    )

    print(
        f"{case_name} | "
        f"target={target_voxel_count} | "
        f"prediction={prediction_voxel_count} | "
        f"state={empty_mask_state}"
    )

Case A | target=0 | prediction=0 | state=both_empty
Case B | target=0 | prediction=1 | state=target_empty_prediction_nonempty
Case C | target=1 | prediction=0 | state=target_nonempty_prediction_empty
Case D | target=1 | prediction=1 | state=both_nonempty


In [8]:
# Cell 2 — Overlap Metric Empty-Mask Policy

def compute_binary_overlap_metrics(
    target_mask: torch.Tensor,       # [D, H, W], torch.bool
    prediction_mask: torch.Tensor,   # [D, H, W], torch.bool
    both_empty_policy: str = "exclude",
) -> tuple[
    torch.Tensor,  # Scalar Dice, shape=[]
    torch.Tensor,  # Scalar IoU, shape=[]
    bool,          # Aggregation 포함 여부
    str,           # Empty-mask state
]:
    """명시적 empty-mask 정책을 적용한 binary Dice와 IoU 계산."""

    # 지원하는 both-empty 정책인지 확인
    if both_empty_policy not in {
        "exclude",
        "perfect",
    }:
        raise ValueError(
            "both_empty_policy는 'exclude' 또는 'perfect'여야 합니다."
        )

    # Target·prediction의 empty-mask 상태와 foreground 크기 확인
    (
        empty_mask_state,
        target_voxel_count,
        prediction_voxel_count,
    ) = classify_binary_mask_pair(
        target_mask=target_mask,
        prediction_mask=prediction_mask,
    )

    # Scalar metric 생성에 사용할 dtype과 device 지정
    metric_options = {
        "dtype": torch.float32,
        "device": target_mask.device,
    }

    # 두 mask가 모두 비어 있을 때 명시적 평가 정책 적용
    if empty_mask_state == "both_empty":
        if both_empty_policy == "exclude":
            dice_score = torch.full(
                size=(),
                fill_value=float("nan"),
                **metric_options,
            )

            iou_score = torch.full(
                size=(),
                fill_value=float("nan"),
                **metric_options,
            )

            include_in_aggregation = False

        else: # perfect
            dice_score = torch.ones(
                size=(),
                **metric_options,
            )

            iou_score = torch.ones(
                size=(),
                **metric_options,
            )

            include_in_aggregation = True

    # 한쪽 mask만 비어 있는 FP-only 또는 FN-only 상태에 0점 부여
    elif empty_mask_state in {
        "target_empty_prediction_nonempty",
        "target_nonempty_prediction_empty",
    }:
        dice_score = torch.zeros(
            size=(),
            **metric_options,
        )

        iou_score = torch.zeros(
            size=(),
            **metric_options,
        )

        include_in_aggregation = True

    # 두 mask 모두 foreground를 포함하는 일반 상태의 overlap 계산
    else:
        intersection_voxel_count = (
            target_mask & prediction_mask
        ).sum().to(torch.float32)

        union_voxel_count = (
            target_mask | prediction_mask
        ).sum().to(torch.float32)

        target_size = torch.tensor(
            target_voxel_count,
            **metric_options,
        )

        prediction_size = torch.tensor(
            prediction_voxel_count,
            **metric_options,
        )

        dice_score = (
            2.0 * intersection_voxel_count
            / (target_size + prediction_size)
        )

        iou_score = (
            intersection_voxel_count
            / union_voxel_count
        )

        include_in_aggregation = True

    return (
        dice_score,
        iou_score,
        include_in_aggregation,
        empty_mask_state,
    )


# 네 가지 empty-mask 조합에 exclude 정책 적용
for (
    case_name,
    (
        target_mask,
        prediction_mask,
    ),
) in empty_mask_test_cases.items():
    (
        dice_score,
        iou_score,
        include_in_aggregation,
        empty_mask_state,
    ) = compute_binary_overlap_metrics(
        target_mask=target_mask,
        prediction_mask=prediction_mask,
        both_empty_policy="exclude",
    )

    print(
        f"{case_name} | "
        f"state={empty_mask_state} | "
        f"Dice={dice_score.item():.3f} | "
        f"IoU={iou_score.item():.3f} | "
        f"include={include_in_aggregation}"
    )


# 동일한 both-empty case에 perfect 정책을 적용하여 차이 확인
perfect_dice, perfect_iou, perfect_include, _ = (
    compute_binary_overlap_metrics(
        target_mask=empty_mask,
        prediction_mask=empty_mask,
        both_empty_policy="perfect",
    )
)

print()
print(
    "Both-empty with perfect policy | "
    f"Dice={perfect_dice.item():.3f} | "
    f"IoU={perfect_iou.item():.3f} | "
    f"include={perfect_include}"
)

Case A | state=both_empty | Dice=nan | IoU=nan | include=False
Case B | state=target_empty_prediction_nonempty | Dice=0.000 | IoU=0.000 | include=True
Case C | state=target_nonempty_prediction_empty | Dice=0.000 | IoU=0.000 | include=True
Case D | state=both_nonempty | Dice=1.000 | IoU=1.000 | include=True

Both-empty with perfect policy | Dice=1.000 | IoU=1.000 | include=True


In [9]:
# Cell 3 — Surface Metric Empty-Mask Policy

def apply_surface_metric_empty_policy(
    empty_mask_state: str,
    nonempty_nsd_score: float | None = None,
    nonempty_hd95_mm: float | None = None,
) -> tuple[
    torch.Tensor,  # Scalar NSD, shape=[]
    bool,          # NSD aggregation 포함 여부
    torch.Tensor,  # Scalar HD95, shape=[]
    bool,          # HD95 aggregation 포함 여부
    bool,          # 한쪽 surface 부재 failure 여부
]:
    """Empty-mask 상태에 따른 NSD·HD95 평가 정책 적용."""

    # Cell 1에서 정의한 네 가지 상태만 허용
    valid_empty_mask_states = {
        "both_empty",
        "target_empty_prediction_nonempty",
        "target_nonempty_prediction_empty",
        "both_nonempty",
    }

    if empty_mask_state not in valid_empty_mask_states:
        raise ValueError(
            f"지원하지 않는 empty-mask 상태: {empty_mask_state}"
        )

    # CPU scalar metric 생성을 위한 공통 설정
    metric_options = {
        "dtype": torch.float32,
    }

    # 두 surface가 모두 없으므로 두 metric 모두 계산 대상에서 제외
    if empty_mask_state == "both_empty":
        nsd_score = torch.full(
            size=(),
            fill_value=float("nan"),
            **metric_options,
        )

        hd95_mm = torch.full(
            size=(),
            fill_value=float("nan"),
            **metric_options,
        )

        include_nsd = False
        include_hd95 = False
        surface_failure = False

    # 한쪽 surface만 존재하는 catastrophic segmentation failure 처리
    elif empty_mask_state in {
        "target_empty_prediction_nonempty",
        "target_nonempty_prediction_empty",
    }:
        # 일치 가능한 양방향 surface가 없으므로 NSD 최저점 부여
        nsd_score = torch.zeros(
            size=(),
            **metric_options,
        )

        # 대응 surface가 없어 HD95 거리 계산 불가능
        hd95_mm = torch.full(
            size=(),
            fill_value=float("nan"),
            **metric_options,
        )

        include_nsd = True
        include_hd95 = False
        surface_failure = True

    # 두 surface가 존재할 때 앞 단계에서 계산한 실제 metric 사용
    else:
        if (
            nonempty_nsd_score is None
            or nonempty_hd95_mm is None
        ):
            raise ValueError(
                "both_nonempty 상태에는 실제 NSD와 HD95가 필요합니다."
            )

        # NSD의 유효 범위 확인
        if not 0.0 <= nonempty_nsd_score <= 1.0:
            raise ValueError(
                "NSD는 0.0 이상 1.0 이하여야 합니다."
            )

        # Physical distance의 음수 값 차단
        if nonempty_hd95_mm < 0.0:
            raise ValueError(
                "HD95는 0 mm 이상이어야 합니다."
            )

        nsd_score = torch.tensor(
            nonempty_nsd_score,
            **metric_options,
        )

        hd95_mm = torch.tensor(
            nonempty_hd95_mm,
            **metric_options,
        )

        include_nsd = True
        include_hd95 = True
        surface_failure = False

    return (
        nsd_score,
        include_nsd,
        hd95_mm,
        include_hd95,
        surface_failure,
    )


# 네 가지 empty-mask 상태에 surface metric 정책 적용
for (
    case_name,
    (
        target_mask,
        prediction_mask,
    ),
) in empty_mask_test_cases.items():
    (
        empty_mask_state,
        _,
        _,
    ) = classify_binary_mask_pair(
        target_mask=target_mask,
        prediction_mask=prediction_mask,
    )

    # 현재 예제의 both-nonempty mask는 서로 동일하므로 완벽한 metric 사용
    if empty_mask_state == "both_nonempty":
        measured_nsd_score = 1.0
        measured_hd95_mm = 0.0
    else:
        measured_nsd_score = None
        measured_hd95_mm = None

    (
        nsd_score,
        include_nsd,
        hd95_mm,
        include_hd95,
        surface_failure,
    ) = apply_surface_metric_empty_policy(
        empty_mask_state=empty_mask_state,
        nonempty_nsd_score=measured_nsd_score,
        nonempty_hd95_mm=measured_hd95_mm,
    )

    print(
        f"{case_name} | "
        f"state={empty_mask_state} | "
        f"NSD={nsd_score.item():.3f} | "
        f"include_nsd={include_nsd} | "
        f"HD95={hd95_mm.item():.3f} mm | "
        f"include_hd95={include_hd95} | "
        f"surface_failure={surface_failure}"
    )

Case A | state=both_empty | NSD=nan | include_nsd=False | HD95=nan mm | include_hd95=False | surface_failure=False
Case B | state=target_empty_prediction_nonempty | NSD=0.000 | include_nsd=True | HD95=nan mm | include_hd95=False | surface_failure=True
Case C | state=target_nonempty_prediction_empty | NSD=0.000 | include_nsd=True | HD95=nan mm | include_hd95=False | surface_failure=True
Case D | state=both_nonempty | NSD=1.000 | include_nsd=True | HD95=0.000 mm | include_hd95=True | surface_failure=False


In [10]:
# Cell 4 — Case/Class Macro Aggregation

def aggregate_case_class_metric(
    metric_values: torch.Tensor,  # [N, C], floating-point
    include_mask: torch.Tensor,   # [N, C], torch.bool
) -> tuple[
    torch.Tensor,  # Per-case macro scores [N]
    torch.Tensor,  # Per-class macro scores [C]
    torch.Tensor,  # Case-first macro score []
    torch.Tensor,  # Class-first macro score []
    torch.Tensor,  # Flat valid-pair mean []
    torch.Tensor,  # Valid case counts per class [C]
]:
    """Case·class metric table의 세 가지 aggregation 계산."""

    # Case와 class axis를 가진 2D metric table인지 확인
    if metric_values.ndim != 2:
        raise ValueError(
            "metric_values의 Shape는 [N, C]여야 합니다."
        )

    # Metric과 include mask의 동일 Shape 확인
    if metric_values.shape != include_mask.shape:
        raise ValueError(
            "metric_values와 include_mask의 Shape가 일치해야 합니다."
        )

    # 평균 계산이 가능한 floating-point metric인지 확인
    if not torch.is_floating_point(metric_values):
        raise TypeError(
            "metric_values는 floating-point Tensor여야 합니다."
        )

    # 명시적인 aggregation 여부를 가진 boolean mask인지 확인
    if include_mask.dtype != torch.bool:
        raise TypeError(
            "include_mask는 torch.bool Tensor여야 합니다."
        )

    # 포함 대상으로 표시된 metric 내부의 NaN·Inf 차단
    included_metric_values = metric_values[
        include_mask
    ]

    if not torch.isfinite(
        included_metric_values
    ).all():
        raise ValueError(
            "포함 대상으로 표시된 metric은 모두 유한해야 합니다."
        )

    # 제외할 case-class pair를 NaN으로 변환
    masked_metric_values = torch.where(
        include_mask,
        metric_values,
        torch.full_like(
            metric_values,
            fill_value=float("nan"),
        ),
    )  # [N, C]

    # Class axis 제거: 환자마다 유효 class의 평균 계산
    per_case_macro_scores = torch.nanmean(
        masked_metric_values,
        dim=1,
    )  # [N]

    # Case axis 제거: class마다 유효 환자의 평균 계산
    per_class_macro_scores = torch.nanmean(
        masked_metric_values,
        dim=0,
    )  # [C]

    # 환자별 평균에 동일한 가중치를 부여한 case-first macro 계산
    case_first_macro_score = torch.nanmean(
        per_case_macro_scores,
    )  # []

    # Class별 평균에 동일한 가중치를 부여한 class-first macro 계산
    class_first_macro_score = torch.nanmean(
        per_class_macro_scores,
    )  # []

    # 모든 유효 case-class pair에 동일한 가중치를 부여한 평균 계산
    flat_valid_pair_mean = torch.nanmean(
        masked_metric_values,
    )  # []

    # 각 class 평균에 실제로 포함된 환자 수 계산
    valid_case_counts = include_mask.sum(
        dim=0,
    )  # [C]

    return (
        per_case_macro_scores,
        per_class_macro_scores,
        case_first_macro_score,
        class_first_macro_score,
        flat_valid_pair_mean,
        valid_case_counts,
    )


# 네 환자와 세 장기로 구성된 가상 Dice table 생성
class_names: list[str] = [
    "liver",
    "kidney",
    "pancreas",
]

dice_values = torch.tensor(
    [
        [0.90, 0.70, float("nan")],
        [0.80, float("nan"), float("nan")],
        [0.85, 0.50, 0.20],
        [0.95, 0.60, 0.10],
    ],
    dtype=torch.float32,
)  # [N=4, C=3]


# NaN이 아닌 case-class pair만 aggregation에 포함
dice_include_mask = torch.isfinite(
    dice_values
)  # [N=4, C=3]


(
    per_case_dice,
    per_class_dice,
    case_first_dice,
    class_first_dice,
    flat_pair_dice,
    valid_case_counts,
) = aggregate_case_class_metric(
    metric_values=dice_values,
    include_mask=dice_include_mask,
)


# 환자별 macro Dice 출력
print("Per-case macro Dice")

for case_index, case_score in enumerate(
    per_case_dice
):
    print(
        f"Case {case_index}: "
        f"{case_score.item():.4f}"
    )


# 장기별 macro Dice와 실제 포함 환자 수 출력
print()
print("Per-class macro Dice")

for (
    class_name,
    class_score,
    valid_case_count,
) in zip(
    class_names,
    per_class_dice,
    valid_case_counts,
):
    print(
        f"{class_name:8s} | "
        f"Dice={class_score.item():.4f} | "
        f"valid cases={valid_case_count.item()}"
    )


# 서로 다른 aggregation 순서가 만드는 최종 점수 차이 출력
print()
print(
    "Case-first macro: ",
    f"{case_first_dice.item():.4f}",
)

print(
    "Class-first macro:",
    f"{class_first_dice.item():.4f}",
)

print(
    "Flat pair mean:   ",
    f"{flat_pair_dice.item():.4f}",
)


# Cell 3의 one-sided empty surface failure 예시 구성
surface_failure_mask = torch.tensor(
    [
        [False, False, False],
        [False, True, False],
        [False, False, True],
        [False, False, False],
    ],
    dtype=torch.bool,
)  # [N=4, C=3]


# Target 또는 prediction surface가 하나라도 존재했던 평가 기회 표시
surface_opportunity_mask = torch.tensor(
    [
        [True, True, False],
        [True, True, False],
        [True, True, True],
        [True, True, True],
    ],
    dtype=torch.bool,
)  # [N=4, C=3]


# Surface가 존재했던 평가 기회 중 one-sided failure 비율 계산
surface_failure_count = (
    surface_failure_mask
    & surface_opportunity_mask
).sum()

surface_opportunity_count = (
    surface_opportunity_mask.sum()
)

surface_failure_rate = (
    surface_failure_count.to(torch.float32)
    / surface_opportunity_count.to(torch.float32)
)  # []


print()
print(
    "Surface failure count:",
    surface_failure_count.item(),
)

print(
    "Surface opportunities:",
    surface_opportunity_count.item(),
)

print(
    "Surface failure rate: ",
    f"{surface_failure_rate.item():.4f}",
)

Per-case macro Dice
Case 0: 0.8000
Case 1: 0.8000
Case 2: 0.5167
Case 3: 0.5500

Per-class macro Dice
liver    | Dice=0.8750 | valid cases=4
kidney   | Dice=0.6000 | valid cases=3
pancreas | Dice=0.1500 | valid cases=2

Case-first macro:  0.6667
Class-first macro: 0.5417
Flat pair mean:    0.6222

Surface failure count: 2
Surface opportunities: 10
Surface failure rate:  0.2000
